# DS-GA 1011 HW4 — Part 2: T5 Text-to-SQL
Run on Google Colab (GPU). Code in `/content/nyu-dsga1011-a4/part2`; `results/`, `records/`, `checkpoints/` were symlinked to Google Drive. Cells collected from my working notebooks, in the order the experiments were run, with original outputs.


## Setup

In [1]:
%cd /content
!rm -rf nyu-dsga1011-a4
!git clone -q https://github.com/gregdurrett/nyu-dsga1011-a4
%cd /content/nyu-dsga1011-a4/part2
!pip install -q wandb sentencepiece
!mkdir -p results records checkpoints
!ls data records

/content
/content/nyu-dsga1011-a4/part2
data:
dev.nl	 flight_database.db	 test.nl   train.sql
dev.sql  flight_database.schema  train.nl

records:
ground_truth_dev.pkl


Link outputs to Drive

In [3]:
import os, shutil
D = "/content/drive/MyDrive/dsga1011/part2_outputs"
P = "/content/nyu-dsga1011-a4/part2"
for name in ["results", "records", "checkpoints"]:
    os.makedirs(f"{D}/{name}", exist_ok=True)
    local = f"{P}/{name}"
    if os.path.islink(local):
        continue
    if os.path.isdir(local):
        for f in os.listdir(local):
            if not os.path.exists(f"{D}/{name}/{f}"):
                shutil.copy(f"{local}/{f}", f"{D}/{name}/{f}")
        shutil.rmtree(local)
    os.symlink(f"{D}/{name}", local)
!ls -la {P} | grep -- '->'

lrwxrwxrwx 1 root root    57 Nov 16 00:20 checkpoints -> /content/drive/MyDrive/dsga1011/part2_outputs/checkpoints
lrwxrwxrwx 1 root root    53 Nov 16 00:20 records -> /content/drive/MyDrive/dsga1011/part2_outputs/records
lrwxrwxrwx 1 root root    53 Nov 16 00:20 results -> /content/drive/MyDrive/dsga1011/part2_outputs/results


## Q4. Data statistics (T5 tokenizer)

In [5]:
%cd /content/nyu-dsga1011-a4/part2
from transformers import T5TokenizerFast
tok = T5TokenizerFast.from_pretrained("google-t5/t5-small")

def load_lines(path):
    with open(path) as f:
        return [line.strip() for line in f]

train_nl  = load_lines("data/train.nl")
train_sql = load_lines("data/train.sql")
dev_nl    = load_lines("data/dev.nl")
dev_sql   = load_lines("data/dev.sql")

/content/nyu-dsga1011-a4/part2


In [8]:
def stats(lines):
    lengths = []
    vocab = set()
    for line in lines:
        ids = tok(line)["input_ids"]      # list of token IDs for this sentence
        lengths.append(len(ids))               # how many tokens? (hint: len)
        vocab.update(ids)                 # add all IDs of this sentence to the set
    mean_len = sum(lengths) / len(lengths)
    return mean_len, len(vocab)             # vocab size = number of distinct IDs

for name, lines in [("train NL", train_nl), ("train SQL", train_sql),
                ("dev NL", dev_nl), ("dev SQL", dev_sql)]:
    mean_len, vocab_size = stats(lines)
    print(f"{name:10s}  examples={len(lines)}  mean_len={mean_len:.2f}  vocab={vocab_size}")

train NL    examples=4225  mean_len=18.10  vocab=792
train SQL   examples=4225  mean_len=217.37  vocab=556
dev NL      examples=466  mean_len=18.07  vocab=466
dev SQL     examples=466  mean_len=211.05  vocab=396


## Data loading (`load_data.py`): `T5Dataset`, `normal_collate_fn`, `test_collate_fn`

In [12]:
%cd /content/nyu-dsga1011-a4/part2
new_block = r'''PAD_IDX = 0
BOS_ID = T5TokenizerFast.from_pretrained("google-t5/t5-small").convert_tokens_to_ids("<extra_id_0>")

class T5Dataset(Dataset):

    def __init__(self, data_folder, split):
        """
        Skeleton for the class for performing data processing for the T5 model.

        Some tips for implementation:
            * You should be using the 'google-t5/t5-small' tokenizer checkpoint to tokenize both
              the encoder and decoder output.
            * You want to provide the decoder some beginning of sentence token. Any extra-id on the
              T5Tokenizer should serve that purpose.
            * Class behavior should be different on the test set.
        """
        # TODO
        self.split = split
        self.tokenizer = T5TokenizerFast.from_pretrained("google-t5/t5-small")
        self.process_data(data_folder, split, self.tokenizer)

    def process_data(self, data_folder, split, tokenizer):
        # TODO
        # Read English questions
        nl_lines = load_lines(os.path.join(data_folder, f"{split}.nl"))

        # Convert English into token IDs
        self.encoder_ids = [tokenizer(x)["input_ids"] for x in nl_lines]

        # Training/dev data have SQL targets
        if split != "test":
            sql_lines = load_lines(os.path.join(data_folder, f"{split}.sql"))

            # Convert SQL into token IDs
            self.decoder_ids = [tokenizer(x)["input_ids"] for x in sql_lines]

            assert len(self.encoder_ids) == len(self.decoder_ids)

    def __len__(self):
        # TODO
        return len(self.encoder_ids)

    def __getitem__(self, idx):
        # TODO
        # Retrieve one English question
        enc = torch.tensor(self.encoder_ids[idx], dtype=torch.long)

        # No SQL targets for test examples
        if self.split == "test":
            return enc

        # Retrieve the matching SQL query
        dec = torch.tensor(self.decoder_ids[idx], dtype=torch.long)

        return enc, dec

def normal_collate_fn(batch):
    """
    Collation function to perform dynamic padding for training and evaluation with the
    development or validation set.

    Inputs:
        * batch (List[Any]): batch is a list of length batch_size, where each index contains what
                             the dataset __getitem__ function returns.

    Returns: To be compatible with the provided training loop, you should be returning
        * encoder_ids: The input ids of shape BxT to be fed into the T5 encoder.
        * encoder_mask: Mask of shape BxT associated with padding tokens in the encoder input
        * decoder_inputs: Decoder input ids of shape BxT' to be fed into T5 decoder.
        * decoder_targets: The target tokens with which to train the decoder (the tokens following each decoder input)
        * initial_decoder_inputs: The very first input token to be decoder (only to be used in evaluation)
    """
    # TODO
    encs = [enc for enc, dec in batch]
    decs = [dec for enc, dec in batch]

    # 1. encoder: pad + mask
    encoder_ids  = pad_sequence(encs, batch_first=True, padding_value=PAD_IDX)
    encoder_mask = (encoder_ids != PAD_IDX).long()

    # 2. decoder: inputs = BOS + all-but-last token, targets = all tokens
    bos = torch.tensor([BOS_ID])
    dec_in = [torch.cat([bos, d[:-1]]) for d in decs]
    decoder_inputs  = pad_sequence(dec_in, batch_first=True, padding_value=PAD_IDX)
    decoder_targets = pad_sequence(decs,   batch_first=True, padding_value=PAD_IDX)

    # 3. first decoder token for each example (used for generation later)
    initial_decoder_inputs = torch.full((len(batch), 1), BOS_ID)

    return encoder_ids, encoder_mask, decoder_inputs, decoder_targets, initial_decoder_inputs

def test_collate_fn(batch):
    """
    Collation function to perform dynamic padding for inference on the test set.

    Inputs:
        * batch (List[Any]): batch is a list of length batch_size, where each index contains what
                             the dataset __getitem__ function returns.

    Recommended returns:
        * encoder_ids: The input ids of shape BxT to be fed into the T5 encoder.
        * encoder_mask: Mask of shape BxT associated with padding tokens in the encoder input
        * initial_decoder_inputs: The very first input token to be decoder (only to be used in evaluation)
    """
    # TODO
    encs = batch                     # each item is already just the encoder tensor

    encoder_ids  = pad_sequence(encs, batch_first=True, padding_value=PAD_IDX)
    encoder_mask = (encoder_ids != PAD_IDX).long()
    initial_decoder_inputs = torch.full((len(batch), 1), BOS_ID)

    return encoder_ids, encoder_mask, initial_decoder_inputs

'''
src = open("load_data.py").read()
s = src.index("PAD_IDX = 0")
e = src.index("def get_dataloader(")
open("load_data.py", "w").write(src[:s] + new_block + src[e:])

import ast; ast.parse(open("load_data.py").read())
print("load_data.py patched — syntax OK")
!grep -n "^class \|^def \|^PAD_IDX\|^BOS_ID\|# TODO" load_data.py

/content/nyu-dsga1011-a4/part2
load_data.py patched — syntax OK
14:PAD_IDX = 0
15:BOS_ID = T5TokenizerFast.from_pretrained("google-t5/t5-small").convert_tokens_to_ids("<extra_id_0>")
17:class T5Dataset(Dataset):
30:        # TODO
36:        # TODO
53:        # TODO
57:        # TODO
70:def normal_collate_fn(batch):
86:    # TODO
105:def test_collate_fn(batch):
118:    # TODO
127:def get_dataloader(batch_size, split):
136:def load_t5_data(batch_size, test_batch_size):
144:def load_lines(path):
150:def load_prompting_data(data_folder):
151:    # TODO


In [13]:
import importlib, load_data; importlib.reload(load_data)
enc, mask, dec_in, dec_tgt, init = next(iter(load_data.get_dataloader(4, "train")))
print(enc.shape, mask.shape, dec_in.shape, dec_tgt.shape, init.shape)
print("dec_in :", dec_in[0, :6].tolist())
print("dec_tgt:", dec_tgt[0, :6].tolist())
print([t.shape for t in next(iter(load_data.get_dataloader(4, "test")))])

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


torch.Size([4, 30]) torch.Size([4, 30]) torch.Size([4, 238]) torch.Size([4, 238]) torch.Size([4, 1])
dec_in : [32099, 3, 23143, 14196, 3, 15438]
dec_tgt: [3, 23143, 14196, 3, 15438, 25424]
[torch.Size([4, 16]), torch.Size([4, 16]), torch.Size([4, 1])]


## `train_t5.py`: `eval_epoch`, `test_inference`

In [17]:
%cd /content/nyu-dsga1011-a4/part2
src = open("train_t5.py").read()

# imports (added once)
imp = "from utils import compute_metrics, save_queries_and_records\n"
extra = ("from transformers import T5TokenizerFast\n"
         "from load_data import BOS_ID\n"
         "TOKENIZER = T5TokenizerFast.from_pretrained(\"google-t5/t5-small\")\n")
if "TOKENIZER =" not in src:
    src = src.replace(imp, imp + extra, 1)

# eval_epoch
old_eval = "    # TODO\n    model.eval()\n    return 0, 0, 0, 0, 0\n"
new_eval = '''    # TODO
    model.eval()
    criterion = nn.CrossEntropyLoss()
    total_loss, total_tokens = 0, 0
    generated = []

    with torch.no_grad():
        for encoder_input, encoder_mask, decoder_input, decoder_targets, initial_decoder_inputs in tqdm(dev_loader):
            encoder_input   = encoder_input.to(DEVICE)
            encoder_mask    = encoder_mask.to(DEVICE)
            decoder_input   = decoder_input.to(DEVICE)
            decoder_targets = decoder_targets.to(DEVICE)

            # 1. loss (same as train_epoch, without backward/step)
            logits = model(input_ids=encoder_input, attention_mask=encoder_mask, decoder_input_ids=decoder_input)['logits']
            non_pad = decoder_targets != PAD_IDX
            loss = criterion(logits[non_pad], decoder_targets[non_pad])
            num_tokens = non_pad.sum().item()
            total_loss += loss.item() * num_tokens
            total_tokens += num_tokens

            # 2. generate SQL
            outputs = model.generate(
                input_ids=encoder_input,
                attention_mask=encoder_mask,
                decoder_start_token_id=BOS_ID,
                max_new_tokens=520,
            )
            generated += TOKENIZER.batch_decode(outputs, skip_special_tokens=True)

    # 3. save queries + run them on the database
    save_queries_and_records(generated, model_sql_path, model_record_path)

    # 4. score
    sql_em, record_em, record_f1, error_msgs = compute_metrics(gt_sql_pth, model_sql_path, gt_record_path, model_record_path)
    error_rate = sum(1 for m in error_msgs if m) / len(error_msgs)

    return total_loss / total_tokens, record_f1, record_em, sql_em, error_rate
'''
assert old_eval in src, "eval_epoch placeholder not found (already patched?)"
src = src.replace(old_eval, new_eval, 1)

# test_inference
s = src.index("def test_inference(")
e = src.index("def main(", s)
fn = src[s:e]
assert "    pass\n" in fn, "test_inference placeholder not found (already patched?)"
fn = fn.replace("    pass\n", '''    # TODO
    model.eval()
    generated = []

    with torch.no_grad():
        for encoder_input, encoder_mask, initial_decoder_inputs in tqdm(test_loader):
            encoder_input = encoder_input.to(DEVICE)
            encoder_mask  = encoder_mask.to(DEVICE)

            outputs = model.generate(
                input_ids=encoder_input,
                attention_mask=encoder_mask,
                decoder_start_token_id=BOS_ID,
                max_new_tokens=520,
            )
            generated += TOKENIZER.batch_decode(outputs, skip_special_tokens=True)

    save_queries_and_records(generated, model_sql_path, model_record_path)
''', 1)
src = src[:s] + fn + src[e:]

open("train_t5.py", "w").write(src)
import ast; ast.parse(src); print("train_t5.py patched — syntax OK")
!grep -n "TOKENIZER =\|BOS_ID\|def eval_epoch\|def test_inference" train_t5.py

/content/nyu-dsga1011-a4/part2
train_t5.py patched — syntax OK
17:from load_data import BOS_ID
18:TOKENIZER = T5TokenizerFast.from_pretrained("google-t5/t5-small")
136:def eval_epoch(args, model, dev_loader, gt_sql_pth, model_sql_path, gt_record_path, model_record_path):
171:                decoder_start_token_id=BOS_ID,
185:def test_inference(args, model, test_loader, model_sql_path, model_record_path):
202:                decoder_start_token_id=BOS_ID,


## `t5_utils.py`: `initialize_model`, `save_model`, `load_model_from_checkpoint`

In [ ]:
%cd /content/nyu-dsga1011-a4/part2
src = open("t5_utils.py").read()

def fill(src, func_name, next_func, code):
    s = src.index(func_name); e = src.index(next_func, s)
    fn = src[s:e]
    assert "    pass\n" in fn, f"{func_name}: already filled?"
    return src[:s] + fn.replace("    pass\n", code, 1) + src[e:]

src = fill(src, "def initialize_model(", "def mkdir(", '''    if args.finetune:
        model = T5ForConditionalGeneration.from_pretrained("google-t5/t5-small")
    else:
        config = T5Config.from_pretrained("google-t5/t5-small")
        model = T5ForConditionalGeneration(config)      # random weights
    return model.to(DEVICE)
''')
src = fill(src, "def save_model(", "def load_model_from_checkpoint(", '''    path = os.path.join(checkpoint_dir, "best" if best else "last")
    mkdir(path)
    model.save_pretrained(path)
''')
src = fill(src, "def load_model_from_checkpoint(", "def initialize_optimizer_and_scheduler(", '''    model_type = 'ft' if args.finetune else 'scr'
    checkpoint_dir = os.path.join('checkpoints', f'{model_type}_experiments', args.experiment_name)
    path = os.path.join(checkpoint_dir, "best" if best else "last")
    model = T5ForConditionalGeneration.from_pretrained(path)
    return model.to(DEVICE)
''')
open("t5_utils.py", "w").write(src)
import ast; ast.parse(src); print("t5_utils.py patched")


In [16]:
import importlib, t5_utils; importlib.reload(t5_utils)
class A: finetune = True; experiment_name = "test"
m = t5_utils.initialize_model(A())
print(type(m).__name__, next(m.parameters()).device)

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  242MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

T5ForConditionalGeneration cuda:0


## Fine-tuning, first model
1-epoch smoke test

In [23]:
%cd /content/nyu-dsga1011-a4/part2
!python3 train_t5.py --finetune --learning_rate 5e-4 --batch_size 16 --max_n_epochs 1 --patience_epochs 1 --experiment_name smoke

/content/nyu-dsga1011-a4/part2
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
Loading weights: 100% 131/131 [00:00<00:00, 4302.56it/s]
100% 265/265 [03:04<00:00,  1.43it/s]
Epoch 0: Average train loss was 0.6432658604837049
100% 30/30 [03:06<00:00,  6.22s/it]
466it [00:07, 64.16it/s]
Epoch 0: Dev loss: 0.1830656117703501, Record F1: 0.2989229403201858, Record EM: 0.259656652360515, SQL EM: 0.0
Epoch 0: 54.72% of the generated outputs led to SQL errors
Writing model shards: 100% 1/1 [00:03<00:00,  3.62s/it]
Writing model shards: 100% 1/1 [00:03<00:00,  3.91s/it]
Loading weights: 100% 131/131 [00:00<00:00, 1916.13it/s]
100% 30/30 [03:00<00:00,  6.02s/it]
466it [00:07, 64.16it/s]
Dev set results: Loss: 0.1830656117703501, Record F1: 0.2989229403201858, Record EM: 0.259656652360515, SQL EM: 0.0
Dev set results: 54.72% of the generated outputs led to SQL errors
100% 27/27 [02:33<00:00,  5.68s/it]
432it [00:08, 51.35it/s]


Full run (`experiment`). Per-epoch dev F1 here is underestimated: the default 120 s DB timeout cut off many queries (`290it [02:00` etc.).

In [24]:
%cd /content/nyu-dsga1011-a4/part2
!python3 train_t5.py --finetune --learning_rate 5e-4 --batch_size 16 --max_n_epochs 20 --patience_epochs 3 --experiment_name experiment

/content/nyu-dsga1011-a4/part2
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
Loading weights: 100% 131/131 [00:00<00:00, 16253.63it/s]
 29% 77/265 [00:53<02:08,  1.47it/s][W1115 04:20:43.106131911 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 1052770304 bytes (free: 593297408, total: 15637086208).
100% 265/265 [03:03<00:00,  1.44it/s]
Epoch 0: Average train loss was 0.5975754139610947
100% 30/30 [02:37<00:00,  5.23s/it]
290it [02:00,  2.42it/s]
Epoch 0: Dev loss: 0.10609113970406994, Record F1: 0.31946663159238614, Record EM: 0.27467811158798283, SQL EM: 0.008583690987124463
Epoch 0: 54.72% of the generated outputs led to SQL errors
Writing model shards: 100% 1/1 [00:04<00:00,  4.67s/it]
Writing model shards: 100% 1/1 [00:02<00:00,  2.96s/it]
100% 265/265 [03:07<00:00,  1.41it/s]
Epoch 1: Average train loss was 0.1221083927736013
100% 30/30 [02:52<00:00,  5.76s/it]
2

Diagnosis: how many dev queries timed out

In [25]:
%cd /content/nyu-dsga1011-a4/part2
import pickle, collections
recs, errs = pickle.load(open("records/t5_ft_experiment_dev.pkl", "rb"))
print(collections.Counter("timeout" if e == "Query timed out" else ("ok" if not e else e.split(":")[0]) for e in errs))

sql  = open("results/t5_ft_experiment_dev.sql").read().splitlines()
gold = open("data/dev.sql").read().splitlines()
lens = [len(q.split()) for q in sql]
print("avg words generated:", sum(lens)/len(lens), "| max:", max(lens), "| gold avg:", sum(len(g.split()) for g in gold)/len(gold))

i = next(k for k, e in enumerate(errs) if e == "Query timed out")
print("\nTIMED OUT example:\nMODEL:", sql[i][:400], "\nGOLD :", gold[i][:400])
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv

/content/nyu-dsga1011-a4/part2
Counter({'ok': 215, 'timeout': 197, 'OperationalError': 54})
avg words generated: 54.24463519313305 | max: 146 | gold avg: 58.896995708154506

TIMED OUT example:
MODEL: SELECT DISTINCT fare_1.fare_id FROM fare fare_1, fare_basis fare_basis_1, days days_1, date_day date_day_1, flight_fare flight_fare_1, flight flight_1, airport_service airport_service_1, city city_1, airport_service airport_service_2, city city_2, days days_2, date_day date_day_2 WHERE fare_1.fare_basis_code = fare_basis_1.fare_basis_code AND fare_basis_1.basis_days = days_1.days_code AND days_1.d 
GOLD : SELECT DISTINCT fare_1.fare_id FROM fare fare_1 , flight_fare flight_fare_1 , flight flight_1 , airport_service airport_service_1 , city city_1 , airport_service airport_service_2 , city city_2 , days days_1 , date_day date_day_1 , fare_basis fare_basis_1 , days days_2 , date_day date_day_2 WHERE fare_1.fare_id = flight_fare_1.fare_id AND flight_fare_1.flight_id = flight_1.flight_id AND f

Re-score the same dev predictions with the timeout raised to 1500 s → real dev F1 72.30

In [26]:
%cd /content/nyu-dsga1011-a4/part2
import importlib, utils, time
src = open("utils.py").read().replace("timeout_secs = 120", "timeout_secs = 1500")
open("utils.py", "w").write(src); importlib.reload(utils)
!grep -n "timeout_secs =" utils.py

qs = open("results/t5_ft_experiment_dev.sql").read().splitlines()
t = time.time()
utils.save_queries_and_records(qs, "results/rescored_dev.sql", "records/rescored_dev.pkl")
sql_em, rec_em, rec_f1, errs = utils.compute_metrics("data/dev.sql", "results/rescored_dev.sql",
                                                     "records/ground_truth_dev.pkl", "records/rescored_dev.pkl")
print(f"F1 {rec_f1:.4f} | Record EM {rec_em:.4f} | SQL EM {sql_em:.4f} | {time.time()-t:.0f}s")
print("timeouts:", sum(e == "Query timed out" for e in errs), "| errors:", sum(bool(e) for e in errs))

/content/nyu-dsga1011-a4/part2
95:    timeout_secs = 1500


466it [21:37,  2.78s/it] 

F1 0.7230 | Record EM 0.6888 | SQL EM 0.0193 | 1298s
timeouts: 0 | errors: 88


## Extra credit: T5 from scratch
Add `--eval_every`, then train from random weights (no `--finetune`).

In [ ]:
%cd /content/nyu-dsga1011-a4/part2
# Extra credit: option to run dev evaluation only every N epochs (generation is slow)
src = open("train_t5.py").read()
anchor = "    parser.add_argument('--use_wandb', action='store_true',"
add = ("    parser.add_argument('--eval_every', type=int, default=1,\n"
       "                        help=\"Run dev evaluation only every N epochs (extra credit: speeds up from-scratch training)\")\n\n")
if "--eval_every" not in src:
    src = src.replace(anchor, add + anchor, 1)
anchor2 = '        print(f"Epoch {epoch}: Average train loss was {tr_loss}")\n'
add2 = ('\n        # extra credit: only evaluate every args.eval_every epochs (always on the last epoch)\n'
        '        last_epoch = (epoch == args.max_n_epochs - 1)\n'
        '        if (epoch + 1) % args.eval_every != 0 and not last_epoch:\n'
        '            save_model(checkpoint_dir, model, best=False)   # keep the latest weights\n'
        '            continue\n\n')
if "last_epoch = (epoch" not in src:
    src = src.replace(anchor2, anchor2 + add2, 1)
open("train_t5.py", "w").write(src)
import ast; ast.parse(src); print("eval_every added")


lr 1e-3: loss rises after warmup, model collapses (stopped)

In [10]:
!python3 train_t5.py --learning_rate 1e-3 --batch_size 16 --max_n_epochs 40 --num_warmup_epochs 2 --patience_epochs 3 --eval_every 5 --experiment_name experiment_ec

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
tokenizer_config.json: 100% 2.32k/2.32k [00:00<00:00, 6.26MB/s]
spiece.model: 100% 792k/792k [00:00<00:00, 81.5MB/s]
tokenizer.json: 100% 1.39M/1.39M [00:00<00:00, 118MB/s]
config.json: 100% 1.21k/1.21k [00:00<00:00, 360kB/s]
100% 265/265 [01:19<00:00,  3.34it/s]
Epoch 0: Average train loss was 2.365584897704487
Writing model shards: 100% 1/1 [00:00<00:00,  3.19it/s]
100% 265/265 [01:21<00:00,  3.26it/s]
Epoch 1: Average train loss was 0.5893349905035541
Writing model shards: 100% 1/1 [00:00<00:00,  2.21it/s]
100% 265/265 [01:19<00:00,  3.31it/s]
Epoch 2: Average train loss was 0.9633300051225872
Writing model shards: 100% 1/1 [00:00<00:00,  2.23it/s]
100% 265/265 [01:20<00:00,  3.30it/s]
Epoch 3: Average train loss was 1.176821189070859
Writing model shards: 100% 1/1 [00:00<00:00,  2.20it/s]
100% 265/265 [01:19<00:00,  3.32it/s]
Epoch 4: Average train loss was 1.4889246412571917
1

lr 3e-4: final run (cell output was lost by mistake; log pasted below)

In [ ]:
!python3 train_t5.py --learning_rate 3e-4 --batch_size 16 --max_n_epochs 40 --num_warmup_epochs 2 --patience_epochs 3 --eval_every 5 --experiment_name experiment_ec

Rename to the submission names and check

In [13]:
%cd /content/nyu-dsga1011-a4/part2
!cp results/t5_scr_experiment_ec_test.sql results/t5_ft_experiment_ec_test.sql
!cp records/t5_scr_experiment_ec_test.pkl records/t5_ft_experiment_ec_test.pkl
import pickle
q = open("results/t5_ft_experiment_ec_test.sql").read().splitlines()
recs, errs = pickle.load(open("records/t5_ft_experiment_ec_test.pkl", "rb"))
print("queries:", len(q), "| records:", len(recs), "| timeouts:", sum(e == "Query timed out" for e in errs), "| errors:", sum(bool(e) for e in errs))

/content/nyu-dsga1011-a4/part2
queries: 432 | records: 432 | timeouts: 0 | errors: 35


## Q7 final model (`experiment_v2`): `<` placeholder fix + longer training
The T5 vocabulary has no `<`. SQL `<=`/`<` are replaced by `le_op`/`lt_op` before tokenization and converted back after generation.

In [4]:
%cd /content/nyu-dsga1011-a4/part2
import ast
ld = open("load_data.py").read()
old = '            sql_lines = load_lines(os.path.join(data_folder, f"{split}.sql"))\n'
new = old + ('            # T5 vocab has no "<" token: replace with placeholders the tokenizer can represent\n'
             '            sql_lines = [s.replace("<=", "le_op").replace("<", "lt_op") for s in sql_lines]\n')
if "lt_op" not in ld:
    assert old in ld, "load_data.py: sql_lines line not found"
    ld = ld.replace(old, new, 1)
open("load_data.py", "w").write(ld); ast.parse(ld)

tr = open("train_t5.py").read()
anchor = 'TOKENIZER = T5TokenizerFast.from_pretrained("google-t5/t5-small")\n'
helper = ('\ndef postprocess_sql(q):\n'
          '    # undo the "<" placeholders used in load_data.py\n'
          '    return q.replace("le_op", "<=").replace("lt_op", "<")\n')
dec_old = "            generated += TOKENIZER.batch_decode(outputs, skip_special_tokens=True)\n"
dec_new = "            generated += [postprocess_sql(q) for q in TOKENIZER.batch_decode(outputs, skip_special_tokens=True)]\n"
if "postprocess_sql" not in tr:
    assert anchor in tr and tr.count(dec_old) == 2, "train_t5.py: anchors not found"
    tr = tr.replace(anchor, anchor + helper, 1).replace(dec_old, dec_new)
open("train_t5.py", "w").write(tr); ast.parse(tr)
print("patched")
!grep -n "lt_op\|postprocess_sql\|timeout_secs =\|--eval_every" load_data.py train_t5.py utils.py

/content/nyu-dsga1011-a4/part2
patched
load_data.py:47:            sql_lines = [s.replace("<=", "le_op").replace("<", "lt_op") for s in sql_lines]
train_t5.py:18:def postprocess_sql(q):
train_t5.py:20:    return q.replace("le_op", "<=").replace("lt_op", "<")
train_t5.py:49:    parser.add_argument('--eval_every', type=int, default=1,
train_t5.py:186:            generated += [postprocess_sql(q) for q in TOKENIZER.batch_decode(outputs, skip_special_tokens=True)]
train_t5.py:217:            generated += [postprocess_sql(q) for q in TOKENIZER.batch_decode(outputs, skip_special_tokens=True)]
utils.py:95:    timeout_secs = 1500


Check that the placeholders survive tokenize → decode

In [5]:
from transformers import T5TokenizerFast
tok = T5TokenizerFast.from_pretrained("google-t5/t5-small")
sql = [l.strip() for l in open("data/train.sql")]
bad = 0
for s in sql:
    if "<" in s:
        p = s.replace("<=", "le_op").replace("<", "lt_op")
        back = tok.decode(tok(p)["input_ids"], skip_special_tokens=True).replace("le_op", "<=").replace("lt_op", "<")
        bad += back.count("<") != s.count("<")
print("queries with <:", sum("<" in s for s in sql), "| round-trip failures:", bad)

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

queries with <: 393 | round-trip failures: 0


Training (stopped after the epoch-15 evaluation; best = epoch 15, dev F1 82.42)

In [6]:
%cd /content/nyu-dsga1011-a4/part2
!python3 train_t5.py --finetune --learning_rate 5e-4 --batch_size 16 --max_n_epochs 20 --num_warmup_epochs 1 --patience_epochs 3 --eval_every 4 --experiment_name experiment_v2

/content/nyu-dsga1011-a4/part2
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
config.json: 100% 1.21k/1.21k [00:00<00:00, 374kB/s]

model.safetensors: downloading bytes:  45% 108M/242M [00:03<00:02, 45.9MB/s, 9.10MB/s  ]
model.safetensors: reconstructing file:  28% 67.0M/242M [00:03<00:09, 18.1MB/s]
model.safetensors: downloading bytes: 100% 108M/108M [00:04<00:00, 23.6MB/s, 9.23MB/s  ]
model.safetensors: reconstructing file: 100% 242M/242M [00:04<00:00, 52.6MB/s, 21.5MB/s  ]
Loading weights: 100% 131/131 [00:00<00:00, 17352.09it/s]
generation_config.json: 100% 147/147 [00:00<00:00, 493kB/s]
  9% 23/265 [00:17<02:18,  1.75it/s][W1116 00:23:13.261697824 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 1046478848 bytes (free: 1031602176, total: 15637086208).
100% 265/265 [02:58<00:00,  1.48it/s]
Epoch 0: Average train loss was 1.8138946562898823
Writing model shards: 100% 1/1 [

Test inference from the best checkpoint

In [7]:
%cd /content/nyu-dsga1011-a4/part2
src = open("utils.py").read().replace("timeout_secs = 1500", "timeout_secs = 3600")
open("utils.py", "w").write(src)
!grep -n "timeout_secs =" utils.py

/content/nyu-dsga1011-a4/part2
95:    timeout_secs = 3600


In [8]:
%%writefile /content/run_test.py
import load_data, t5_utils, train_t5
class A: finetune = True; experiment_name = "experiment_v2"
model = t5_utils.load_model_from_checkpoint(A(), best=True)
test_loader = load_data.get_dataloader(16, "test")
train_t5.test_inference(A(), model, test_loader, "results/t5_ft_experiment_test.sql", "records/t5_ft_experiment_test.pkl")
print("DONE")

Writing /content/run_test.py


In [12]:
!cd /content/nyu-dsga1011-a4/part2 && PYTHONPATH=. python3 /content/run_test.py

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
Loading weights: 100% 131/131 [00:00<00:00, 2053.97it/s]
100% 27/27 [01:45<00:00,  3.91s/it]
432it [02:59,  2.40it/s]
DONE


In [13]:
import pickle
q = open("results/t5_ft_experiment_test.sql").read().splitlines()
recs, errs = pickle.load(open("records/t5_ft_experiment_test.pkl", "rb"))
print("queries:", len(q), "| records:", len(recs), "| timeouts:", sum(e == "Query timed out" for e in errs), "| errors:", sum(bool(e) for e in errs), "| contains '<':", sum("<" in x for x in q))

queries: 432 | records: 432 | timeouts: 0 | errors: 49 | contains '<': 40
